# Model tuning — Modelado-Pneumonia (Google Colab)

Clones the repository, installs dependencies, and runs `scripts/tune_models.py` / `scripts/compare_models.py` against the shared remote database (`results_unsa_ira`) — the same one used from a local machine. No local state to lose: each search persists progress to the database as it runs.

**Resumable by design**: if a Colab session disconnects mid-search, rerunning the same cell with the same parameters picks up where it left off. Any trial already persisted is detected and reused instantly (never recomputed); only the trial in flight at the moment of disconnection needs to rerun.

## 1. Clone the repository and install dependencies

In [ ]:
REPO_URL = "https://github.com/<org>/Modelado-Pneumonia.git"
BRANCH = "feature/modelling-basics"

!git clone --branch "$BRANCH" "$REPO_URL" repo
%cd repo
!pip install -q -r requirements.txt

## 2. Database credentials

Store the connection string as a Colab Secret (key icon 🔑 in the left panel) named `DATABASE_URL`, with the same value used in the project's local `.env` file, and grant this notebook access to it. Do not paste credentials directly into a cell.

In [ ]:
import os
from google.colab import userdata

os.environ["DATABASE_URL"] = userdata.get("DATABASE_URL")

# Optional: override the project-wide random seed for this session (default is
# already 42). Only needed for a deliberate seed-robustness check.
# os.environ["RANDOM_SEED"] = "42"

## 3. Sanity check (database connectivity + project import)

In [ ]:
from pneumonia.data.load_data import get_db_engine
from sqlalchemy import text

eng = get_db_engine(None)
with eng.connect() as conn:
    n = conn.execute(text("SELECT COUNT(*) FROM results_unsa_ira.walkforward_runs")).scalar()
print(f"Connection OK — {n} runs already stored in the shared database.")

## 4. Tune one model/department

Edit the parameters below and run the cell. If the session disconnects partway through, rerunning this same cell resumes automatically.

In [ ]:
DEPARTMENT = "AMAZONAS"
MODEL = "SARIMA"          # SARIMA, RandomForest, XGBoost, Prophet, HoltWinters, LSTM, GRU
SEARCH_METHOD = "optuna"  # optuna (recommended for larger search spaces), grid (small spaces, e.g. HoltWinters), random
N_ITER = 30                # ignored when SEARCH_METHOD=grid
AGE_GROUP = "under5"       # under5 | 60plus

!python scripts/tune_models.py \
    --department "$DEPARTMENT" \
    --model "$MODEL" \
    --age_group "$AGE_GROUP" \
    --search_method "$SEARCH_METHOD" \
    --n_iter "$N_ITER"

## 5. Batch run across departments/models

Each (department, model) combination is independently resumable — if the session is interrupted, rerunning this cell skips whatever already completed (detected almost instantly, no recomputation) and continues with what remains.

In [ ]:
import subprocess

DEPARTMENTS = ["AMAZONAS", "LIMA", "CUSCO"]
MODELS = ["SARIMA", "XGBoost"]
SEARCH_METHOD = "optuna"
N_ITER = 30
AGE_GROUP = "under5"

failed = []
for dept in DEPARTMENTS:
    for model in MODELS:
        print(f"\n{'='*70}\n{dept} / {model}\n{'='*70}")
        result = subprocess.run([
            "python", "scripts/tune_models.py",
            "--department", dept, "--model", model, "--age_group", AGE_GROUP,
            "--search_method", SEARCH_METHOD, "--n_iter", str(N_ITER),
        ])
        if result.returncode != 0:
            failed.append((dept, model))

if failed:
    print(f"\nFailed: {failed}")
else:
    print("\nAll combinations completed without errors.")

## 6. View results (protected holdout number, by default)

In [ ]:
!python scripts/compare_models.py --department "$DEPARTMENT" --source db